# Support Vector Machine — From Scratch
> Hard-margin linear SVM using Sub-Gradient Descent.

**Hinge loss:** L = λ‖w‖² + (1/n) Σ max(0, 1 − yᵢ(wᵀxᵢ + b))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
np.random.seed(42)

## SVM Class (Sub-Gradient Descent)

In [ ]:
class SVM:
    def __init__(self, lr=0.001, lambda_param=0.01, n_iters=1000):
        self.lr = lr; self.lam = lambda_param; self.n_iters = n_iters
        self.w = None; self.b = None
        self.losses = []

    def fit(self, X, y):
        # Labels must be -1 / +1
        y_ = np.where(y <= 0, -1, 1)
        n, d = X.shape
        self.w = np.zeros(d); self.b = 0.0
        for _ in range(self.n_iters):
            margins = y_ * (X @ self.w + self.b)
            mask = margins < 1
            dw = 2*self.lam*self.w - (X[mask] * y_[mask, None]).mean(axis=0) if mask.any() else 2*self.lam*self.w
            db = -y_[mask].mean() if mask.any() else 0
            self.w -= self.lr * dw
            self.b -= self.lr * db
            loss = self.lam*np.dot(self.w,self.w) + np.maximum(0, 1-margins).mean()
            self.losses.append(loss)

    def predict(self, X):
        return np.sign(X @ self.w + self.b).astype(int)

## Train & Evaluate

In [ ]:
X, y = make_classification(n_samples=500, n_features=2, n_redundant=0,
                            n_clusters_per_class=1, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
sc = StandardScaler(); X_tr = sc.fit_transform(X_tr); X_te = sc.transform(X_te)

svm = SVM(lr=0.001, lambda_param=0.01, n_iters=1000)
svm.fit(X_tr, y_tr)
print(f"Accuracy: {accuracy_score(y_te, svm.predict(X_te))*100:.2f}%")

## Decision Boundary & Loss Curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Decision boundary
h=0.05
xx, yy = np.meshgrid(np.arange(X_tr[:,0].min()-1, X_tr[:,0].max()+1, h),
                     np.arange(X_tr[:,1].min()-1, X_tr[:,1].max()+1, h))
Z = svm.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
axes[0].contourf(xx, yy, Z, alpha=0.3, cmap='RdBu')
axes[0].scatter(X_tr[:,0], X_tr[:,1], c=y_tr, cmap='RdBu', edgecolors='k', s=30)
axes[0].set_title("SVM Decision Boundary")

# Support hyperplanes
w, b = svm.w, svm.b
x_line = np.linspace(X_tr[:,0].min(), X_tr[:,0].max(), 100)
for margin, ls in [(0,'r-'), (1,'r--'), (-1,'r--')]:
    y_line = -(w[0]*x_line + b - margin) / (w[1]+1e-10)
    axes[0].plot(x_line, y_line, ls, lw=1.5)

# Loss
axes[1].plot(svm.losses, color='tomato')
axes[1].set_title("Hinge Loss"); axes[1].set_xlabel("Iteration")
plt.tight_layout(); plt.show()